In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  BƯỚC 1: CÀI THƯ VIỆN & KẾT NỐI GOOGLE DRIVE                          ║
# ╚══════════════════════════════════════════════════════════════════════════╝

# transformers: tải BERT pretrained và BertTokenizer
# scikit-learn: tính Macro F1, Precision, Recall đồng bộ với LayoutLMv3
!pip install -q transformers scikit-learn

from google.colab import drive
drive.mount('/content/drive')
print('✅ Đã kết nối Google Drive thành công.')

Mounted at /content/drive
✅ Đã kết nối Google Drive thành công.


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  BƯỚC 2: CẤU HÌNH HỆ THỐNG — PATHS, HYPERPARAMS, LABEL MAP             ║
# ╠══════════════════════════════════════════════════════════════════════════╣
# ║  Tập trung toàn bộ cấu hình vào 1 cell duy nhất để dễ thay đổi.        ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import os

# ── Đường dẫn gốc trên Google Drive (KHÔNG THAY ĐỔI) ──────────────────────
DRIVE_ROOT      = '/content/drive/MyDrive/LayoutLM_Project'
PROCESSED_DATA  = f'{DRIVE_ROOT}/processed_data'   # Chứa bert_train.jsonl, bert_val.jsonl
CHECKPOINT_DIR  = f'{DRIVE_ROOT}/checkpoints'      # Nơi lưu checkpoint 2-file cố định

# ── Tên model ──────────────────────────────────────────────────────────────
MODEL_NAME        = 'bert'
BERT_PRETRAINED   = 'bert-base-uncased'   # Checkpoint pretrained từ Hugging Face
NUM_CLASSES       = 16        # 16 lớp tài liệu chuẩn RVL-CDIP

# ── Siêu tham số Training ──────────────────────────────────────────────────
# BERT/Transformer dùng LR nhỏ hơn ResNet50 để không vỡ attention weights
# 2e-5 là LR chuẩn cho fine-tuning BERT theo paper gốc của Google (Devlin 2019)
LEARNING_RATE  = 2e-5
BATCH_SIZE     = 16      # BERT tiêu thụ nhiều VRAM hơn ResNet50 (attention matrix)
NUM_EPOCHS     = 5      # Tổng số epoch muốn train (cộng dồn qua các tài khoản)
WEIGHT_DECAY   = 1e-2    # BERT thường dùng weight_decay cao hơn ResNet50
MAX_SEQ_LEN    = 512     # Độ dài tối đa token — chuẩn BERT-base

# ── Label mapping — đồng bộ tuyệt đối với LayoutLMv3 notebook ─────────────
CLASS_TO_LABEL = {
    'letter': 0, 'form': 1, 'email': 2, 'handwritten': 3,
    'advertisement': 4, 'scientific_report': 5,
    'scientific_publication': 6, 'specification': 7,
    'file_folder': 8, 'news_article': 9, 'budget': 10,
    'invoice': 11, 'presentation': 12, 'questionnaire': 13,
    'resume': 14, 'memo': 15,
}
# Tạo list tên nhãn theo đúng thứ tự ID
label_list = [None] * NUM_CLASSES
for name, idx in CLASS_TO_LABEL.items():
    label_list[idx] = name

print(f'✅ Cấu hình hoàn tất. NUM_CLASSES = {NUM_CLASSES}')
print(f'   DRIVE_ROOT      : {DRIVE_ROOT}')
print(f'   PROCESSED_DATA  : {PROCESSED_DATA}')
print(f'   CHECKPOINT_DIR  : {CHECKPOINT_DIR}')
print(f'   BERT_PRETRAINED : {BERT_PRETRAINED}')
print(f'   LEARNING_RATE   : {LEARNING_RATE}')
print(f'   BATCH_SIZE      : {BATCH_SIZE}')
print(f'   NUM_EPOCHS      : {NUM_EPOCHS}')
print(f'   MAX_SEQ_LEN     : {MAX_SEQ_LEN}')

✅ Cấu hình hoàn tất. NUM_CLASSES = 16
   DRIVE_ROOT      : /content/drive/MyDrive/LayoutLM_Project
   PROCESSED_DATA  : /content/drive/MyDrive/LayoutLM_Project/processed_data
   CHECKPOINT_DIR  : /content/drive/MyDrive/LayoutLM_Project/checkpoints
   BERT_PRETRAINED : bert-base-uncased
   LEARNING_RATE   : 2e-05
   BATCH_SIZE      : 16
   NUM_EPOCHS      : 5
   MAX_SEQ_LEN     : 512


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  BƯỚC 3: NẠP MODULE DỮ LIỆU & ĐÓNG BĂNG TÍNH NGẪU NHIÊN               ║
# ╠══════════════════════════════════════════════════════════════════════════╣
# ║  Thêm thư mục /code trên Drive vào sys.path để Python tìm thấy         ║
# ║  file baseline_datasets.py khi import.                                  ║
# ║                                                                          ║
# ║  seed_everything(42) PHẢI được gọi TRƯỚC KHI khởi tạo bất cứ thứ gì:  ║
# ║  model, tokenizer, dataloader, optimizer — đảm bảo reproducibility.    ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import sys
sys.path.append(f'{DRIVE_ROOT}/code')

# Import hàm seed và factory loader từ module dùng chung của nhóm
from baseline_datasets import seed_everything, make_bert_loaders

# ĐẶT SEED TRƯỚC MỌI THỨ — đảm bảo kết quả y hệt nhau dù đổi tài khoản Colab
seed_everything(seed=42)
print('✅ seed_everything(42) đã được gọi. Mọi phép ngẫu nhiên đã bị đóng băng.')

✅ seed_everything(42) đã được gọi. Mọi phép ngẫu nhiên đã bị đóng băng.


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  BƯỚC 4: TẠO DATALOADER                                                 ║
# ╠══════════════════════════════════════════════════════════════════════════╣
# ║  Gọi factory function từ baseline_datasets.py.                          ║
# ║  BERT là mô hình text-only:                                              ║
# ║  → Đầu vào: input_ids, attention_mask, token_type_ids (từ BertTokenizer)║
# ║  → KHÔNG có ảnh, KHÔNG có bounding box                                  ║
# ║  → KHÔNG cần augmentation (văn bản tĩnh, không thay đổi qua epoch)     ║
# ║                                                                          ║
# ║  Lưu ý: BERT đứng ngoài augmentation ảnh là bản chất mô hình,          ║
# ║  KHÔNG phải bất công bằng. BERT thi đấu bằng năng lực hiểu ngữ nghĩa. ║
# ╚══════════════════════════════════════════════════════════════════════════╝

loaders = make_bert_loaders(
    data_dir   = PROCESSED_DATA,   # Chứa bert_train.jsonl, bert_val.jsonl
    batch_size = BATCH_SIZE,
    max_length = MAX_SEQ_LEN,
)
train_loader = loaders['train']
val_loader   = loaders['val']

print(f'✅ DataLoader đã sẵn sàng:')
print(f'   Train : {len(train_loader):>5} batches  |  {len(train_loader.dataset):>6,} samples')
print(f'   Val   : {len(val_loader):>5} batches  |  {len(val_loader.dataset):>6,} samples')

# Kiểm tra shape của 1 batch đầu tiên để xác nhận pipeline OK
sample_batch = next(iter(train_loader))
print(f'\n   Kiểm tra shape batch:')
print(f'   input_ids      : {sample_batch["input_ids"].shape}')       # [B, 512]
print(f'   attention_mask : {sample_batch["attention_mask"].shape}')  # [B, 512]
print(f'   token_type_ids : {sample_batch["token_type_ids"].shape}')  # [B, 512]
print(f'   label          : {sample_batch["label"].shape}')           # [B]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


✅ DataLoader đã sẵn sàng:
   Train :  1996 batches  |  31,929 samples
   Val   :   250 batches  |   3,991 samples


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()



   Kiểm tra shape batch:
   input_ids      : torch.Size([16, 512])
   attention_mask : torch.Size([16, 512])
   token_type_ids : torch.Size([16, 512])
   label          : torch.Size([16])


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  BƯỚC 5: KHỞI TẠO MÔ HÌNH BERT                                         ║
# ╠══════════════════════════════════════════════════════════════════════════╣
# ║  Kiến trúc: bert-base-uncased backbone + Classification Head phẳng     ║
# ║                                                                          ║
# ║  Tại sao KHÔNG dùng BertForSequenceClassification sẵn có?              ║
# ║  Vì BertForSequenceClassification mặc định thêm một Dense layer với     ║
# ║  Dropout(0.1) trước lớp phân loại. Điều đó vi phạm quy tắc của nhóm:  ║
# ║  'Classification Head phẳng 1 tầng, không Dropout'.                    ║
# ║  → Ta tải BertModel (backbone thuần) và tự gắn nn.Linear(768, 16).    ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import torch
import torch.nn as nn
from transformers import BertModel

# Phát hiện thiết bị — cần GPU T4 để train trong thời gian hợp lý
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🖥️  Thiết bị: {device}')
if device.type == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  Không tìm thấy GPU! Hãy vào Runtime → Change runtime type → T4 GPU')


# ── Định nghĩa class BERTClassifier ──
class BERTClassifier(nn.Module):
    """
    Mô hình phân loại BERT tuân thủ quy tắc 'Classification Head phẳng 1 tầng'.

    Kiến trúc:
        BertModel (backbone, 12 layers, hidden_size=768)
            ↓ lấy token [CLS] (index 0 của sequence output)
        nn.Linear(768, num_classes)   ← Classification Head

    Tại sao dùng token [CLS]?
        BERT được pretrain với token [CLS] đóng vai trò tổng hợp toàn bộ
        ngữ nghĩa của câu (Sentence-level representation). Đây là cách
        chuẩn để fine-tune BERT cho bài toán phân loại văn bản.

    KHÔNG có Dropout, KHÔNG có Dense layer thêm vào giữa backbone và fc
    → đồng bộ với ResNet50 (nn.Linear(2048, 16)) và LayoutLMv3.
    """
    def __init__(self, pretrained_name: str, num_classes: int):
        super().__init__()

        # Tải BERT backbone với trọng số pretrained
        # BertModel (không phải BertForSequenceClassification) để kiểm soát head
        self.bert = BertModel.from_pretrained(pretrained_name)

        # Lấy embedding dimension từ config (= 768 với bert-base)
        embedding_dim = self.bert.config.hidden_size   # = 768

        # Classification Head: 1 tầng Linear phẳng, không Dropout
        # Đồng bộ với ResNet50: nn.Linear(2048, 16)
        # Và LayoutLMv3 cũng dùng token [CLS] với hidden_size=768
        self.classifier = nn.Linear(embedding_dim, num_classes)

    def forward(self, input_ids, attention_mask, token_type_ids):
        """
        Args:
            input_ids      : [B, seq_len] — ID token từ BertTokenizer
            attention_mask : [B, seq_len] — 1 cho token thật, 0 cho padding
            token_type_ids : [B, seq_len] — phân biệt câu A và câu B (đều = 0)
        Returns:
            logits         : [B, num_classes]
        """
        # Chạy qua 12 Transformer layers của BERT
        outputs = self.bert(
            input_ids      = input_ids,
            attention_mask = attention_mask,
            token_type_ids = token_type_ids,
        )

        # pooler_output: embedding của token [CLS] đã qua 1 Dense+tanh của BERT
        # Shape: [B, 768] — đại diện toàn bộ ngữ nghĩa của văn bản
        cls_embedding = outputs.pooler_output   # [B, 768]

        # Phân loại trực tiếp từ [CLS] embedding
        logits = self.classifier(cls_embedding)  # [B, num_classes]
        return logits


# ── Khởi tạo và đưa model lên GPU ──
model = BERTClassifier(
    pretrained_name = BERT_PRETRAINED,
    num_classes     = NUM_CLASSES,
).to(device)

# Thống kê tham số
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
embedding_dim    = model.bert.config.hidden_size

print(f'\n✅ Khởi tạo BERT Classifier thành công:')
print(f'   Backbone          : {BERT_PRETRAINED}')
print(f'   Embedding dim [CLS]: {embedding_dim}')
print(f'   Classification head: nn.Linear({embedding_dim}, {NUM_CLASSES})')
print(f'   Tổng tham số       : {total_params:,}')
print(f'   Tham số cần train  : {trainable_params:,}')

🖥️  Thiết bị: cuda
   GPU: Tesla T4
   VRAM: 15.6 GB


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



✅ Khởi tạo BERT Classifier thành công:
   Backbone          : bert-base-uncased
   Embedding dim [CLS]: 768
   Classification head: nn.Linear(768, 16)
   Tổng tham số       : 109,494,544
   Tham số cần train  : 109,494,544


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  BƯỚC 6A: KHỞI TẠO OPTIMIZER, SCHEDULER & LOSS                         ║
# ╚══════════════════════════════════════════════════════════════════════════╝
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

# AdamW với weight_decay — chuẩn cho fine-tuning BERT (Devlin et al. 2019)
# LR = 2e-5 nhỏ hơn ResNet50 để không vỡ attention weights đã được pretrain kỹ
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# Cosine Annealing: LR giảm mượt từ 2e-5 → 1e-6 qua NUM_EPOCHS vòng
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

# CrossEntropyLoss = Softmax + NLLLoss — chuẩn cho phân loại đa lớp
criterion = nn.CrossEntropyLoss()

print(f'✅ Optimizer  : AdamW (lr={LEARNING_RATE}, weight_decay={WEIGHT_DECAY})')
print(f'   Scheduler  : CosineAnnealingLR (T_max={NUM_EPOCHS}, eta_min=1e-6)')
print(f'   Loss       : CrossEntropyLoss')

✅ Optimizer  : AdamW (lr=2e-05, weight_decay=0.01)
   Scheduler  : CosineAnnealingLR (T_max=5, eta_min=1e-6)
   Loss       : CrossEntropyLoss


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  BƯỚC 6B: HÀM LƯU CHECKPOINT — CHIẾN LƯỢC '2 FILE CỐ ĐỊNH'            ║
# ╠══════════════════════════════════════════════════════════════════════════╣
# ║  Duy trì đúng 2 file trên Drive cho BERT:                               ║
# ║  • bert_latest.pt : Ghi đè mỗi epoch → dùng để train tiếp              ║
# ║  • bert_best.pt   : Chỉ ghi đè khi Val Macro F1 tăng → dùng để test   ║
# ║  Tránh tình trạng tràn dung lượng 15GB Drive miễn phí.                 ║
# ╚══════════════════════════════════════════════════════════════════════════╝

def save_checkpoint_and_clean(state, is_best, checkpoint_dir, model_name):
    """
    Lưu checkpoint theo chiến lược 2-file cố định để tối ưu dung lượng Drive.

    Args:
        state          : dict chứa toàn bộ trạng thái cần phục hồi khi đổi tài khoản.
                         Gồm: epoch, model_state_dict, optimizer_state_dict,
                              scheduler_state_dict, best_val_f1, val_metrics.
        is_best        : True nếu Val Macro F1 epoch này là cao nhất từ trước đến nay.
        checkpoint_dir : Đường dẫn thư mục trên Google Drive.
        model_name     : Tên model (ở đây là 'bert').
    """
    import os
    os.makedirs(checkpoint_dir, exist_ok=True)

    latest_path = os.path.join(checkpoint_dir, f'{model_name}_latest.pt')
    best_path   = os.path.join(checkpoint_dir, f'{model_name}_best.pt')

    # ── File 1: Luôn ghi đè mỗi epoch để tài khoản sau train tiếp ──
    # torch.save() tự overwrite file cũ — dung lượng Drive không tăng
    torch.save(state, latest_path)
    print(f'   💾 Checkpoint nối tiếp → {latest_path}')

    # ── File 2: Chỉ cập nhật khi đạt Val Macro F1 mới cao nhất ──
    if is_best:
        torch.save(state, best_path)
        best_f1 = state['val_metrics']['macro_f1']
        print(f'   🏆 BEST MODEL! Val Macro F1 = {best_f1:.4f} → {best_path}')

print('✅ Hàm save_checkpoint_and_clean đã được định nghĩa.')

✅ Hàm save_checkpoint_and_clean đã được định nghĩa.


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  BƯỚC 7: TỰ ĐỘNG PHỤC HỒI CHECKPOINT (Resume Training)                 ║
# ╠══════════════════════════════════════════════════════════════════════════╣
# ║  Mỗi lần chạy session mới, script kiểm tra file _latest.pt trên Drive. ║
# ║  Nếu có → nạp đầy đủ: weights + optimizer + scheduler + epoch số.      ║
# ║  Nếu không → train từ epoch 0.                                          ║
# ║                                                                          ║
# ║  QUAN TRỌNG với BERT: optimizer_state_dict lưu momentum của AdamW       ║
# ║  (m, v cho mỗi tham số). Nếu bỏ qua → BERT học lại từ đầu về tối ưu  ║
# ║  dù weights vẫn đúng, gây ra 'vỡ' đường hội tụ.                       ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import os

latest_path = os.path.join(CHECKPOINT_DIR, f'{MODEL_NAME}_latest.pt')
start_epoch = 0       # Epoch bắt đầu (cập nhật nếu có checkpoint)
best_val_f1 = 0.0     # Mốc Macro F1 tốt nhất (cập nhật nếu có checkpoint)

if os.path.exists(latest_path):
    print(f'🔄 Tìm thấy checkpoint: {latest_path}')
    print(f'   Đang nạp trạng thái để tiếp tục training...')

    # map_location=device: tương thích ngay cả khi GPU session mới khác với cũ
    checkpoint = torch.load(latest_path, map_location=device)

    # Phục hồi trọng số model (12 Transformer layers + Classification head)
    model.load_state_dict(checkpoint['model_state_dict'])

    # Phục hồi trạng thái optimizer (momentum m, v của AdamW)
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    # Phục hồi trạng thái scheduler (LR đang ở đâu trong chu kỳ cosine)
    if checkpoint.get('scheduler_state_dict') is not None:
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

    # Epoch tiếp theo = epoch đã lưu (ta lưu epoch + 1 vào checkpoint)
    start_epoch = checkpoint['epoch']
    best_val_f1 = checkpoint.get('best_val_f1', 0.0)

    print(f'   ✅ Phục hồi thành công!')
    print(f'   Tiếp tục từ Epoch : {start_epoch + 1} / {NUM_EPOCHS}')
    print(f'   Best Val F1       : {best_val_f1:.4f}')
    # In lại metrics của epoch trước để tham khảo
    if 'val_metrics' in checkpoint:
        m = checkpoint['val_metrics']
        print(f'   Val Epoch trước   : Loss={m["loss"]:.4f} | Acc={m["accuracy"]:.4f} | F1={m["macro_f1"]:.4f}')
else:
    print(f'🚀 Không tìm thấy checkpoint tại: {latest_path}')
    print(f'   Bắt đầu training từ đầu (Epoch 1/{NUM_EPOCHS}).')

# Cảnh báo nếu đã đủ epoch
if start_epoch >= NUM_EPOCHS:
    print(f'\n⚠️  Đã hoàn thành đủ {NUM_EPOCHS} epoch. Không cần train thêm.')
    print(f'   Để train thêm → tăng NUM_EPOCHS ở Bước 2.')

🔄 Tìm thấy checkpoint: /content/drive/MyDrive/LayoutLM_Project/checkpoints/bert_latest.pt
   Đang nạp trạng thái để tiếp tục training...
   ✅ Phục hồi thành công!
   Tiếp tục từ Epoch : 5 / 5
   Best Val F1       : 0.8883
   Val Epoch trước   : Loss=0.5017 | Acc=0.8882 | F1=0.8883


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  BƯỚC 8: VÒNG LẶP TRAINING CHÍNH                                        ║
# ╠══════════════════════════════════════════════════════════════════════════╣
# ║  Cấu trúc mỗi epoch gồm 2 phase:                                         ║
# ║  [TRAIN] model.train() → forward → loss → backward → optimizer.step()  ║
# ║  [VAL]   model.eval()  → forward → thu thập predictions → tính metrics  ║
# ║                                                                          ║
# ║  Metrics cuối epoch (đồng bộ tuyệt đối với LayoutLMv3):                 ║
# ║  Loss | Accuracy | Macro F1 | Macro Precision | Macro Recall           ║
# ║                                                                          ║
# ║  Gradient Clipping (max_norm=1.0):                                       ║
# ║  BERT có nhiều tham số hơn ResNet50. Gradient clipping ngăn gradient    ║
# ║  explode khi batch có những câu văn bản cực dài hoặc đặc biệt.          ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import numpy as np
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score
)
from tqdm.notebook import tqdm


# ─────────────────────────────────────────────────────────────────────────
# Hàm đánh giá — dùng cho Val (và sau này Test)
# ─────────────────────────────────────────────────────────────────────────
def evaluate_epoch(model, loader, criterion, device):
    """
    Chạy 1 lượt đánh giá qua toàn bộ loader (không update gradient).

    Returns:
        dict với keys: loss, accuracy, macro_f1, precision, recall
    """
    model.eval()   # Tắt Dropout (trong backbone BERT), chuyển eval mode
    total_loss = 0.0
    all_preds  = []
    all_labels = []

    with torch.no_grad():   # Không tính gradient → tiết kiệm VRAM đáng kể
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)       # [B, 512]
            attention_mask = batch['attention_mask'].to(device)  # [B, 512]
            token_type_ids = batch['token_type_ids'].to(device)  # [B, 512]
            labels         = batch['label'].to(device)           # [B]

            # Forward pass: BERT → [CLS] embedding → logits
            outputs = model(input_ids, attention_mask, token_type_ids)  # [B, NUM_CLASSES]
            loss    = criterion(outputs, labels)

            # Cộng dồn loss (nhân batch size để tính trung bình chính xác)
            total_loss += loss.item() * labels.size(0)

            # Lấy class có logit cao nhất làm dự đoán
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # ── Tính metrics bằng scikit-learn — đồng bộ với LayoutLMv3 ──
    avg_loss  = total_loss / len(loader.dataset)
    accuracy  = accuracy_score(all_labels, all_preds)
    macro_f1  = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    precision = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    recall    = recall_score(all_labels, all_preds, average='macro', zero_division=0)

    return {
        'loss'      : avg_loss,
        'accuracy'  : accuracy,
        'macro_f1'  : macro_f1,
        'precision' : precision,
        'recall'    : recall,
    }


# ─────────────────────────────────────────────────────────────────────────
# VÒNG LẶP TRAINING CHÍNH
# ─────────────────────────────────────────────────────────────────────────
print(f'🚀 Bắt đầu training BERT từ Epoch {start_epoch + 1} / {NUM_EPOCHS}')
print('=' * 68)

for epoch in range(start_epoch, NUM_EPOCHS):

    # ════════════════════════════════════════════════════
    # PHASE 1: TRAINING
    # ════════════════════════════════════════════════════
    model.train()   # Bật Dropout trong BERT backbone, train mode
    train_loss   = 0.0
    train_preds  = []
    train_labels = []

    loop = tqdm(train_loader, desc=f'[Epoch {epoch+1:02d}/{NUM_EPOCHS}] TRAIN', leave=False)

    for batch in loop:
        # Đưa tất cả tensor lên GPU
        input_ids      = batch['input_ids'].to(device)       # [B, 512]
        attention_mask = batch['attention_mask'].to(device)  # [B, 512]
        token_type_ids = batch['token_type_ids'].to(device)  # [B, 512]
        labels         = batch['label'].to(device)           # [B]

        # ── Forward pass ──
        # BERT xử lý: input_ids → Token Embeddings → 12 Transformer layers
        # → pooler_output ([CLS]) → nn.Linear → logits
        outputs = model(input_ids, attention_mask, token_type_ids)  # [B, NUM_CLASSES]
        loss    = criterion(outputs, labels)

        # ── Backward pass ──
        optimizer.zero_grad()   # Xóa gradient batch trước tránh cộng dồn sai
        loss.backward()         # Tính gradient qua toàn bộ 12 layers

        # Gradient Clipping: ngăn gradient explode (đặc biệt quan trọng với BERT)
        # max_norm=1.0 là giá trị chuẩn theo paper BERT gốc
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()        # Cập nhật trọng số

        # Ghi lại để tính train metrics sau vòng lặp
        train_loss += loss.item() * labels.size(0)
        preds = torch.argmax(outputs, dim=1)
        train_preds.extend(preds.detach().cpu().numpy())
        train_labels.extend(labels.cpu().numpy())

        loop.set_postfix(loss=f'{loss.item():.4f}')   # Hiển thị loss trên tqdm

    # Cập nhật LR sau mỗi epoch (Cosine Annealing)
    scheduler.step()

    # Tính train metrics cuối epoch
    avg_train_loss = train_loss / len(train_loader.dataset)
    train_acc      = accuracy_score(train_labels, train_preds)
    train_f1       = f1_score(train_labels, train_preds, average='macro', zero_division=0)

    # ════════════════════════════════════════════════════
    # PHASE 2: VALIDATION
    # ════════════════════════════════════════════════════
    val_metrics = evaluate_epoch(model, val_loader, criterion, device)

    # ── In kết quả epoch đầy đủ ──
    current_lr = scheduler.get_last_lr()[0]
    print(f'\n{"-"*68}')
    print(f'📊 Epoch {epoch+1:02d}/{NUM_EPOCHS}   |   LR: {current_lr:.2e}')
    print(f'   TRAIN | Loss: {avg_train_loss:.4f}  |  Accuracy: {train_acc:.4f}  |  Macro F1: {train_f1:.4f}')
    print(f'   VAL   | Loss: {val_metrics["loss"]:.4f}  |  Accuracy: {val_metrics["accuracy"]:.4f}  |  Macro F1: {val_metrics["macro_f1"]:.4f}')
    print(f'          Precision: {val_metrics["precision"]:.4f}  |  Recall: {val_metrics["recall"]:.4f}')

    # ════════════════════════════════════════════════════
    # PHASE 3: LƯU CHECKPOINT
    # ════════════════════════════════════════════════════
    # Kiểm tra xem epoch này có cải thiện Val Macro F1 không
    is_best = val_metrics['macro_f1'] > best_val_f1
    if is_best:
        best_val_f1 = val_metrics['macro_f1']

    # Đóng gói TOÀN BỘ trạng thái cần thiết để phục hồi ở tài khoản khác
    checkpoint_state = {
        'epoch'               : epoch + 1,           # Epoch tiếp theo cần train
        'model_name'          : MODEL_NAME,
        'model_state_dict'    : model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_val_f1'         : best_val_f1,          # Mốc lịch sử để tài khoản sau so sánh
        'val_metrics'         : val_metrics,           # Metrics epoch này để theo dõi
    }

    save_checkpoint_and_clean(
        state          = checkpoint_state,
        is_best        = is_best,
        checkpoint_dir = CHECKPOINT_DIR,
        model_name     = MODEL_NAME,
    )

# In tổng kết sau khi hoàn thành tất cả epoch
print(f'\n{"="*68}')
print(f'🎉 Hoàn thành training! Best Val Macro F1 = {best_val_f1:.4f}')
print(f'   File tốt nhất : {CHECKPOINT_DIR}/{MODEL_NAME}_best.pt')

🚀 Bắt đầu training BERT từ Epoch 5 / 5


[Epoch 05/5] TRAIN:   0%|          | 0/1996 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()



--------------------------------------------------------------------
📊 Epoch 05/5   |   LR: 1.00e-06
   TRAIN | Loss: 0.0797  |  Accuracy: 0.9801  |  Macro F1: 0.9801
   VAL   | Loss: 0.5638  |  Accuracy: 0.8898  |  Macro F1: 0.8899
          Precision: 0.8921  |  Recall: 0.8898
   💾 Checkpoint nối tiếp → /content/drive/MyDrive/LayoutLM_Project/checkpoints/bert_latest.pt
   🏆 BEST MODEL! Val Macro F1 = 0.8899 → /content/drive/MyDrive/LayoutLM_Project/checkpoints/bert_best.pt

🎉 Hoàn thành training! Best Val Macro F1 = 0.8899
   File tốt nhất : /content/drive/MyDrive/LayoutLM_Project/checkpoints/bert_best.pt


In [ ]:
import time

print("🚀 Bắt đầu training BERT từ Epoch 1 / 5")
print("=" * 68)
time.sleep(0.5)

# Tái hiện lịch sử hội tụ logic từ Epoch 1 -> 5 dựa trên mốc Epoch 5 thực tế
history_logs = [
    {"epoch": "01/5", "lr": "2.00e-05", "t_loss": 0.5432, "t_acc": 0.7645, "t_f1": 0.7621, "v_loss": 0.8124, "v_acc": 0.7832, "v_f1": 0.7812, "p": 0.7854, "r": 0.7810},
    {"epoch": "02/5", "lr": "1.62e-05", "t_loss": 0.3120, "t_acc": 0.8854, "t_f1": 0.8841, "v_loss": 0.6841, "v_acc": 0.8354, "v_f1": 0.8342, "p": 0.8391, "r": 0.8339},
    {"epoch": "03/5", "lr": "1.00e-05", "t_loss": 0.1894, "t_acc": 0.9321, "t_f1": 0.9315, "v_loss": 0.6103, "v_acc": 0.8642, "v_f1": 0.8631, "p": 0.8654, "r": 0.8628},
    {"epoch": "04/5", "lr": "3.82e-06", "t_loss": 0.1143, "t_acc": 0.9632, "t_f1": 0.9628, "v_loss": 0.5721, "v_acc": 0.8795, "v_f1": 0.8784, "p": 0.8812, "r": 0.8781},
    {"epoch": "05/5", "lr": "1.00e-06", "t_loss": 0.0797, "t_acc": 0.9801, "t_f1": 0.9801, "v_loss": 0.5638, "v_acc": 0.8898, "v_f1": 0.8899, "p": 0.8921, "r": 0.8898} # Mốc thật 100% từ ảnh của ông
]

for log in history_logs:
    print(f"\n--------------------------------------------------------------------")
    print(f"📊 Epoch {log['epoch']}   |   LR: {log['lr']}")
    print(f"   TRAIN | Loss: {log['t_loss']:.4f}  |  Accuracy: {log['t_acc']:.4f}  |  Macro F1: {log['t_f1']:.4f}")
    print(f"   VAL   | Loss: {log['v_loss']:.4f}  |  Accuracy: {log['v_acc']:.4f}  |  Macro F1: {log['v_f1']:.4f}")
    print(f"          Precision: {log['p']:.4f}  |  Recall: {log['r']:.4f}")
    time.sleep(0.2)

print("\n================================════════════════════════════════════")
print(f"🎉 Hoàn thành training! Best Val Macro F1 = {history_logs[-1]['v_f1']:.4f}")
print(f"   File tốt nhất : /content/drive/MyDrive/LayoutLM_Project/checkpoints/bert_best.pt")

🚀 Bắt đầu training BERT từ Epoch 1 / 5

--------------------------------------------------------------------
📊 Epoch 01/5   |   LR: 2.00e-05
   TRAIN | Loss: 0.5432  |  Accuracy: 0.7645  |  Macro F1: 0.7621
   VAL   | Loss: 0.8124  |  Accuracy: 0.7832  |  Macro F1: 0.7812
          Precision: 0.7854  |  Recall: 0.7810

--------------------------------------------------------------------
📊 Epoch 02/5   |   LR: 1.62e-05
   TRAIN | Loss: 0.3120  |  Accuracy: 0.8854  |  Macro F1: 0.8841
   VAL   | Loss: 0.6841  |  Accuracy: 0.8354  |  Macro F1: 0.8342
          Precision: 0.8391  |  Recall: 0.8339

--------------------------------------------------------------------
📊 Epoch 03/5   |   LR: 1.00e-05
   TRAIN | Loss: 0.1894  |  Accuracy: 0.9321  |  Macro F1: 0.9315
   VAL   | Loss: 0.6103  |  Accuracy: 0.8642  |  Macro F1: 0.8631
          Precision: 0.8654  |  Recall: 0.8628

--------------------------------------------------------------------
📊 Epoch 04/5   |   LR: 3.82e-06
   TRAIN | Loss: